# Flux LoRA Training — Colab T4

Тренировка кастомной LoRA конкретного человека с помощью **ai-toolkit** (Ostris) на бесплатном T4 GPU.

**Что вы получите:** LoRA файл (.safetensors), который можно использовать в ComfyUI с нодой LoraLoader для генерации изображений человека в любом стиле/окружении.

**Время:** ~30-60 мин тренировки на T4
**VRAM:** ~14 GB пиковое потребление

---

### Как это работает (пошагово)

```
[1. GPU] → [2. HF Токен] → [3. Установка] → [4. ЗАГРУЗКА ФОТО] → [5. Тренировка] → [6. Тест] → [7. Скачать]
                                                     ↑
                                             Самый важный шаг!
                                        Загрузите 15-30 фото человека
                                        прямо сюда через кнопку upload
```

### Какие фото нужны для тренировки?

| Что нужно | Зачем |
|-----------|-------|
| **15-30 фото** одного человека | Чем больше, тем лучше модель запомнит лицо |
| **Разные ракурсы** (анфас, 3/4, профиль) | Модель учит лицо со всех сторон |
| **Разное освещение** (улица, помещение, студия) | Модель не привязывается к одному свету |
| **Разные выражения** (улыбка, серьёзное, смех) | Естественные результаты в генерации |
| **Чёткое лицо, хорошее качество** | Мусор на входе = мусор на выходе |

### Чего НЕ нужно

- Групповые фото (только один человек на фото!)
- Фото в солнечных очках / маске (лицо должно быть видно)
- Размытые или очень тёмные фото
- Фото только с одного ракурса

**Запускайте ячейки 1-7 по порядку.**

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Авторизация Hugging Face (обязательно для FLUX)
#@markdown ---
#@markdown ### Зачем это нужно?
#@markdown Модель **FLUX.1-dev** от Black Forest Labs — **закрытая модель** на Hugging Face.
#@markdown Для скачивания нужно:
#@markdown 1. Зарегистрироваться на [huggingface.co](https://huggingface.co)
#@markdown 2. Принять лицензию на странице модели: [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
#@markdown 3. Создать токен: [Settings → Access Tokens](https://huggingface.co/settings/tokens) (тип: **Read**)
#@markdown 4. Вставить токен ниже

hf_token = "" #@param {type:"string"}
#@markdown Вставьте ваш Hugging Face токен (начинается с `hf_...`)

if not hf_token or not hf_token.strip().startswith("hf_"):
    print("=" * 60)
    print("  ОШИБКА: Токен не указан или неверный формат!")
    print()
    print("  1. Перейдите: https://huggingface.co/settings/tokens")
    print("  2. Создайте токен с доступом Read")
    print("  3. Вставьте его в поле hf_token выше")
    print("  4. Перезапустите эту ячейку")
    print("=" * 60)
    raise ValueError("Hugging Face токен обязателен для FLUX.1-dev")

!pip install huggingface_hub -q
from huggingface_hub import login
login(token=hf_token.strip())

import os
os.environ["HF_TOKEN"] = hf_token.strip()

print("\n" + "=" * 60)
print("  Авторизация успешна!")
print("  Теперь можно скачивать FLUX.1-dev")
print("=" * 60)

In [ ]:
#@title 3. Установка ai-toolkit + Зависимости
%cd /content
!git clone https://github.com/ostris/ai-toolkit.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ai-toolkit
!git submodule update --init --recursive
!pip install -r requirements.txt -q
!pip install peft accelerate safetensors PyYAML bitsandbytes -q

# Фиксируем numpy < 2.0 — нужно для совместимости с dctorch, numba, tensorflow в Colab
# Также обновляем scipy и jedi до совместимых версий
!pip install "numpy>=1.26,<2.0" scipy jedi -q

print("\nai-toolkit установлен!")

In [ ]:
#@title 4. Загрузка тренировочных фото
#@markdown ## ⬇️ Нажмите кнопку "Выбрать файлы" ниже
#@markdown
#@markdown Загрузите **15-30 фотографий** человека, на которого будете тренировать LoRA.
#@markdown
#@markdown Фото сохранятся в папку `/content/dataset/` — именно оттуда
#@markdown тренировочный скрипт будет брать данные на следующем шаге.
#@markdown
#@markdown **Форматы:** .jpg, .jpeg, .png, .webp
#@markdown
#@markdown ---
#@markdown
#@markdown ### Чек-лист перед загрузкой:
#@markdown - ✅ На каждом фото **один человек**
#@markdown - ✅ Лицо **хорошо видно** (без очков, масок)
#@markdown - ✅ Есть фото с **разных ракурсов** (анфас, 3/4, профиль)
#@markdown - ✅ Есть фото при **разном освещении**
#@markdown - ✅ Фото **чёткие**, не размытые

from google.colab import files
from IPython.display import display, Image as IPImage
import os

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

print("=" * 60)
print("  ЗАГРУЗКА ФОТО ДЛЯ ТРЕНИРОВКИ LoRA")
print("=" * 60)
print(f"\n  Папка датасета: {DATASET_DIR}")
print("  Нажмите кнопку ниже и выберите 15-30 фото\n")

uploaded = files.upload()

VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp')
saved = 0
for name, data in uploaded.items():
    if not name.lower().endswith(VALID_EXT):
        print(f"  Пропущено (неподдерживаемый формат): {name}")
        continue
    path = os.path.join(DATASET_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    saved += 1

# Показываем итог
all_files = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(VALID_EXT)]
count = len(all_files)

print(f"\n{'=' * 60}")
print(f"  Загружено: {saved} фото")
print(f"  Всего в датасете: {count} фото")
print(f"  Папка: {DATASET_DIR}")
print(f"{'=' * 60}")

if count < 10:
    print("\n  ⚠️  Менее 10 фото — результат может быть плохим.")
    print("  Рекомендуется: 15-30 фото для качественной LoRA.")
elif count <= 30:
    print(f"\n  ✅ Отлично! {count} фото — хороший датасет.")
else:
    print(f"\n  ℹ️  {count} фото — много, тренировка может занять дольше.")

# Превью загруженных фото
print(f"\nПревью (первые 8 фото):")
for f in sorted(all_files)[:8]:
    path = os.path.join(DATASET_DIR, f)
    print(f"  📷 {f}")
    try:
        display(IPImage(filename=path, width=200))
    except Exception:
        pass

In [ ]:
#@title 5. Настройка и запуск тренировки
#@markdown Задайте параметры тренировки ниже.

trigger_word = "ohwx" #@param {type:"string"}
#@markdown Триггер-слово — уникальный токен, который активирует LoRA. Используйте что-то редкое, например "ohwx" или "sks".

steps = 1500 #@param {type:"integer"}
#@markdown Количество шагов тренировки (рекомендуется 1000-2000 для T4)

learning_rate = 1e-4 #@param {type:"number"}
resolution = 512 #@param [512, 768] {type:"raw"}
lora_rank = 16 #@param [8, 16, 32] {type:"raw"}
output_name = "my_person_lora" #@param {type:"string"}

import yaml, os, gc, torch

# --- Проверка HF токена ---
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("=" * 60)
    print("  ОШИБКА: HF_TOKEN не найден!")
    print("  Запустите ячейку 2 (Авторизация HF) заново.")
    print("=" * 60)
    raise ValueError("HF_TOKEN не установлен — запустите ячейку 2")

# Сохраняем токен на диск, чтобы subprocess ai-toolkit его подхватил
from huggingface_hub import login
login(token=hf_token)
print("HF токен активен и сохранён на диск.")

# --- Освобождаем GPU память перед тренировкой ---
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU память освобождена. Всего VRAM: {vram_gb:.1f} GiB")

# Борьба с фрагментацией памяти CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

config = {
    "job": "extension",
    "config": {
        "name": output_name,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "performance_log_every": 100,
            "device": "cuda:0",
            "trigger_word": trigger_word,
            "network": {
                "type": "lora",
                "linear": lora_rank,
                "linear_alpha": lora_rank
            },
            "save": {
                "dtype": "float16",
                "save_every": 500,
                "max_step_saves_to_keep": 2
            },
            "datasets": [{
                "folder_path": "/content/dataset",
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [resolution, resolution]
            }],
            "train": {
                "batch_size": 1,
                "steps": steps,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": learning_rate,
                "ema_config": {"use_ema": False},
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True,
                "quantize_device": "cpu"
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 500,
                "width": resolution,
                "height": resolution,
                "prompts": [
                    f"a photo of {trigger_word} person, professional headshot, studio lighting",
                    f"a photo of {trigger_word} person, casual outdoor portrait, natural light"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 3.5,
                "sample_steps": 25
            }
        }]
    }
}

config_path = "/content/ai-toolkit/config/train_lora.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Конфиг сохранён: {config_path}")
print(f"\nПараметры тренировки:")
print(f"  Триггер-слово: {trigger_word}")
print(f"  Шаги: {steps}")
print(f"  Скорость обучения: {learning_rate}")
print(f"  Разрешение: {resolution}x{resolution}")
print(f"  LoRA rank: {lora_rank}")
print(f"  Квантизация: NF4 на CPU (экономия VRAM)")
print(f"\nЗапуск тренировки... (это займёт 30-60 мин на T4)")

# Явно передаём токены и настройку CUDA аллокатора в subprocess
!cd /content/ai-toolkit && \
    HF_TOKEN="{hf_token}" \
    HUGGING_FACE_HUB_TOKEN="{hf_token}" \
    PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python run.py config/train_lora.yaml

print("\nТренировка завершена!")

In [ ]:
#@title 6. Тест LoRA — Генерация тестового фото
#@markdown Быстрый тест обученной LoRA через пайплайн diffusers.

trigger_word = "ohwx" #@param {type:"string"}
prompt = "a photo of ohwx person, professional headshot, studio lighting, 4k" #@param {type:"string"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, os

# Находим последний LoRA файл
lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден! Убедитесь, что тренировка завершена.")
else:
    lora_path = lora_files[-1]
    print(f"Используется LoRA: {lora_path}")

    import torch
    from diffusers import FluxPipeline

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("HF_TOKEN не найден! Запустите ячейку 2 (Авторизация HF) заново.")
        raise ValueError("HF_TOKEN не установлен")

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.bfloat16,
        token=hf_token
    ).to("cuda")
    pipe.load_lora_weights(lora_path)

    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=3.5,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    image.save("/content/test_result.png")

    from IPython.display import display, Image
    display(Image("/content/test_result.png"))
    print("Тестовое изображение сохранено: /content/test_result.png")

    del pipe
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
#@title 7. Скачивание LoRA / Сохранение на Google Drive
#@markdown Выберите, куда сохранить обученный LoRA файл.

save_to_drive = True #@param {type:"boolean"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, shutil, os

lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("LoRA файл не найден!")
else:
    lora_path = lora_files[-1]
    lora_filename = os.path.basename(lora_path)
    print(f"LoRA файл: {lora_path} ({os.path.getsize(lora_path) / 1024**2:.1f} MB)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = "/content/drive/MyDrive/ComfyUI_LoRAs"
        os.makedirs(drive_dir, exist_ok=True)
        dest = f"{drive_dir}/{lora_filename}"
        shutil.copy2(lora_path, dest)
        print(f"Сохранено на Drive: {dest}")
    else:
        from google.colab import files
        files.download(lora_path)
        print("Скачивание начато!")

---
## Использование LoRA в ComfyUI

### Способ 1: Нода LoraLoader
1. Скопируйте LoRA файл в `ComfyUI/models/loras/`
2. В вашем воркфлоу добавьте ноду **LoraLoader**
3. Подключите её между загрузчиком модели и KSampler
4. Выберите ваш LoRA файл
5. Используйте триггер-слово в промпте (например, "a photo of ohwx person in a garden")

### Способ 2: В Colab
Используйте ячейку "Скачать LoRA" в `colab_flux_photo.ipynb` для прямой загрузки.

### Советы
- **Триггер-слово** должно присутствовать в промпте для активации LoRA
- **Сила (strength)** 0.7-1.0 лучше всего работает для LoRA лиц
- **Пониженная сила** (0.4-0.6) для большего стилистического разнообразия
- Комбинируйте с IPAdapter FaceID для ещё лучшей консистентности

### Решение проблем
- **Переобучение (overfit)** (все изображения выглядят одинаково): уменьшите количество шагов, увеличьте разнообразие датасета
- **Недообучение (underfit)** (лицо не распознаётся): увеличьте количество шагов, улучшите качество изображений
- **Ошибки OOM**: уменьшите разрешение до 512, уменьшите batch_size до 1